# data

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cdist
from math import floor, log10


def sig_figs(x: float, precision: int):
    """
    Rounds a number to number of significant figures
    Parameters:
    - x - the number to be rounded
    - precision (integer) - the number of significant figures
    Returns:
    - float
    """

    x = float(x)
    precision = int(precision)

    return round(x, -int(floor(log10(abs(x)))) + (precision - 1))

In [ ]:
def sparse_adjacency(locations, threshold):
    """
    Computes a sparse adjacency matrix from location data.

    Args:
        locations (pandas.DataFrame): DataFrame with x, y (or more) coordinate columns.
        threshold (float): Distance threshold for defining neighbors.

    Returns:
        csr_matrix: Sparse adjacency matrix.
        distances: Pairwise distances between all points.
    """
    # Compute pairwise distances
    distances = squareform(pdist(locations.values))
    
    # Apply threshold to find neighbors
    rows, cols = np.where((distances <= threshold) & (distances != 0))
    
    # Construct sparse adjacency matrix
    data = np.ones(len(rows))
    adj_matrix_sparse = csr_matrix((data, (rows, cols)), shape=distances.shape)

    return adj_matrix_sparse, distances

# --- Load data ---
data_path = str(dataset_dir('starmap', 'BZ5'))
x_data_name = "normalized_data.csv"
index_col = None
celltype_data = pd.read_csv(f"{data_path}/celltype.csv", index_col=index_col)  # Assuming first column is index
celltype_data.columns = ["cell_type"] 
x_data = pd.read_csv(f"{data_path}/{x_data_name}", index_col=index_col)
pos_data = pd.read_csv(f"{data_path}/pos.csv", index_col=index_col)
pos_data.columns = ['x', 'y']
domain_data = pd.read_csv(f"{data_path}/domain.csv", index_col=index_col)
domain_data.columns = ["niche_truth"]

x_data.index = x_data.index.astype(str)
pos_data.index = pos_data.index.astype(str)
celltype_data.index = celltype_data.index.astype(str)
domain_data.index = domain_data.index.astype(str)

# --- Compute adjacency matrix ---
r = 700
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
neighbor_count = pd.DataFrame(neighbor_count.toarray().astype(int), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))

# --- Save results ---
# neighbor_count.to_csv(f"{data_path}/neighbor_count.python.csv")

In [ ]:
# calculate the median distance of neighbors for each cell based on their type
# --- Calculate neighbor distances ---
def calculate_neighbor_distances(adj_matrix, distances, one_hot_matrix):
    """
    Calculates the median distance of neighbors for each cell type.

    Args:
        adj_matrix (csr_matrix): Sparse adjacency matrix.
        distances (np.ndarray): Pairwise distances between all cells.
        one_hot_matrix (csr_matrix): One-hot encoded cell type matrix.

    Returns:
        pd.DataFrame: DataFrame containing median neighbor distances per cell type.
    """
    num_cells = adj_matrix.shape[0]
    cell_types = one_hot_matrix.shape[1]  # Number of different cell types
    median_distances = np.full((num_cells, cell_types), np.nan)  # Initialize all as NaN

    # Loop over each cell
    for i in range(num_cells):
        # Find neighbors of cell `i`
        neighbors = adj_matrix[i].nonzero()[1]  # Indices of neighbors
        
        # If no neighbors, skip to the next cell
        if len(neighbors) == 0:
            continue
        
        # Loop over each cell type
        for cell_type_idx in range(cell_types):
            # Find neighbors of this type (non-zero values in the one-hot matrix)
            type_neighbors = neighbors[one_hot_matrix[neighbors, cell_type_idx].toarray().flatten() > 0]
            
            # If no neighbors of this type, skip
            if len(type_neighbors) == 0:
                median_distances[i, cell_type_idx] = np.nan
            else:
                # Compute median distance of neighbors of this type
                median_distances[i, cell_type_idx] = np.median(distances[i, type_neighbors])
 in the form
    # Convert to DataFrame for easier access
    median_distances_df = pd.DataFrame(median_distances, index=celltype_data.index, columns=one_hot_df.columns.str.lstrip('_'))
    
    return median_distances_df

# Compute the neighbor distances
neighbor_distance = calculate_neighbor_distances(adj_matrix, distances, one_hot_matrix)


In [ ]:
domain_mapping = {0 : "Layer 1", 1 : "Layer 2/3", 2 : "Layer 5", 3 : "Layer 6"}
n_domain = len(domain_mapping)
tissue_region = "mPFC region of mice"

cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}



# process output
test5 batch id : batch_66f46c25e3bc8190ad161e5e3d2cfc29
BZ5 count : batch_66f478846dd8819081ccede49b02690f
BZ5 distance : batch_66f478a9c92881909408801fe45cca7e

In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np
import json
client = OpenAI()

data_name="BZ5"
model_type = "embeddings"  # embeddings end2end
folder_path=f"./batch_json/{data_name}_{model_type}" 
output_path = f"./batch_results/{data_name}_{model_type}"

# parameters
use_full_name = True
with_self_type = True
with_region_name = True
Graph_type = "distance"  #distance or count
with_negatives = True
with_CoT = False

output_file_name = f"{output_path}/response_{data_name}_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}.txt"


In [ ]:
# manually retrieve batch output
batch_id = "batch_66f478a9c92881909408801fe45cca7e"
file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# Open the file in write mode and save the string
with open(output_file_name, 'w') as file:
    file.write(file_response.text)

In [ ]:
# 初始化空列表以保存custom_id和content
data = []
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{data_name}_{i}_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}.txt"
    output_file_name = f"{output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                embedding = json_data['response']['body']['data'][0]['embedding']

                # 将提取到的信息添加到列表中
                data.append({'custom_id': custom_id, 'embedding': embedding})
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
# 提取 custom_id 和 embedding
custom_ids = [item['custom_id'] for item in data]
embeddings = [item['embedding'] for item in data]

# 创建 DataFrame，custom_id 作为 index，embedding 作为列
embeddings_df = pd.DataFrame(embeddings, index=custom_ids)

# clustering

In [ ]:
import scanpy as sc
import anndata as ad
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import KMeans



In [ ]:
adata_embeddings = ad.AnnData(embeddings_df)
adata_embeddings.obs = adata_embeddings.obs.join([celltype_data, domain_data, pos_data])

In [ ]:
res = 0.01
sc.pp.neighbors(adata_embeddings, n_neighbors=50)
sc.tl.leiden(adata_embeddings,directed=False,resolution=res, key_added=f"leiden_{res}")
sc.pl.scatter(adata_embeddings,x="x",y="y", color=f"leiden_{res}")

In [ ]:

adjusted_rand_score(adata_embeddings.obs["niche_truth"], adata_embeddings.obs[f"leiden_{res}"])

In [ ]:
k = 4  
kmeans = KMeans(n_clusters=k, random_state=42)
adata_embeddings.obs['Kmeans'] = kmeans.fit_predict(adata_embeddings.X)

In [ ]:
sc.pl.scatter(adata_embeddings,x="x",y="y", color="Kmeans")

In [ ]:
adjusted_rand_score(adata_embeddings.obs["niche_truth"], adata_embeddings.obs["Kmeans"])